### Building a Fast Plotting Module

In [ ]:
StockPoint_Clustering_and_Routing/src/viz/sp_territory_map.py

## TO-DOs

In [ ]:
# # SELECT 
# #     * ,
#     CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 1
#         WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 2
#         WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 3
#     ELSE 99 END AS assignment_type_id,
#     CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 'Assigned Active/Buying'
#         WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 'Unassigned Active/Buying'
#         WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 'Assigned Recently Activated'
#     ELSE 'Others' END AS assignment_type
# # FROM customer_stockpoint_cluster_assignment 
# # WHERE status = 'ACTIVE'


# # Add assignment_type_id to db table

# A. Data Preparation  
**Data Required**
1. SP Coverage Boundary   
2. Cluster/Beat  
3. Assignment:   
    - Mapped Customer - Active  
    - Mapped Customer - Dormant   


In [ ]:

import duckdb
import pandas as pd
import geopandas as gpd
import pickle
import ast 
from config.settings import EXPORTS_DIR, OUTPUT_DIR, STORAGE_CONFIG,PROCESSED_DATA_DIR

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

In [ ]:
test_plot_dir = OUTPUT_DIR / 'map/tests'
test_plot_dir_boundary = test_plot_dir/ 'boundary'  
test_plot_dir_grid_coverage = test_plot_dir/ 'grid-coverage'  
test_plot_dir_customer_assignment = test_plot_dir/ 'customer-assignment'  


test_spids_contiguous = ['1647131', '1647113','1647127']
test_spids_noncontiguous = ['1646971','1647136','1647376','1647380']

### 0. Base Data

In [ ]:
processed_sp_dim_filepath = PROCESSED_DATA_DIR / 'processed_sp_dim_df.pickle'

with open(processed_sp_dim_filepath, 'rb') as f:
    processed_sp_dim_df = pickle.load(f)

In [ ]:
processed_sp_dim_df.head(1)

### 1. SP Coverage Boundary

In [ ]:
## Load Territory/Coverage Boundary 

# Load Clustering Result
current_date = '2025-08-15' #datetime.now().strftime('%Y-%m-%d')
suffix = "ALL" 
resolution = 8
cluster_results_filepath = EXPORTS_DIR / 'clustering' / f"{suffix}_SPS_CLUSTER_R{resolution}_{current_date}.pickle"

with open(cluster_results_filepath, 'rb') as f:
    cluster_results = pickle.load(f)
    
# cluster_results.keys() # ['territories', 'grid_results', 'assignments', 'statistics', 'territory_version']
sp_territories_dict = cluster_results.get('territories')
sp_clusters_dict = cluster_results.get('grid_results')
sp_customer_assignments_dict = cluster_results.get('assignments')    
    
#  with duckdb.connect(H3_DUCKDB_PATH) as conn:

In [ ]:
# Preprocessed Clipped Cells
all_cluster_grid_list = []

for key, value in sp_clusters_dict.items():
    # print(type(value.get('clipped_cells')))
    cell_geometries = value.get('cell_geometries')  
    if cell_geometries:
        cell_geometries_gpd = pd.DataFrame.from_dict(cell_geometries, orient='index').reset_index()
        cell_geometries_gpd.columns = ['h3_cell','geometry']
        cell_geometries_gpd = gpd.GeoDataFrame(cell_geometries_gpd)
        cell_geometries_gpd['stock_point_id'] = int(key)
        
        all_cluster_grid_list.append(cell_geometries_gpd)
    
# Concatenate only if list is not empty
all_cluster_clipped_grid_list_df = (
    pd.concat(all_cluster_grid_list, ignore_index=True)
    if all_cluster_grid_list
    else gpd.GeoDataFrame(columns=['h3_cell', 'geometry', 'stock_point_id'])
)

### 2. Customer Assignement

In [ ]:
# customer_stockpoint_cluster_assignment_df

with duckdb.connect(H3_DUCKDB_PATH) as conn: 
    customer_stockpoint_cluster_assignment_df = conn.execute('''
                         SELECT 
                            stock_point_id,	a.customer_id,	h3_cell_id,	customer_type,	previous_cluster_id,
                            CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 1
                                WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 2
                                WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 3
                            ELSE 99 END AS assignment_type_id,
                            CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 'Assigned Active/Buying'
                                WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 'Unassigned Active/Buying'
                                WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 'Assigned Recently Activated'
                            ELSE 'Others' END AS assignment_type,
                            ---,	assignment_type_id, assignment_type,	
                            contact_name,state_name, town_name, city_name, latitude, longitude, kyc_capture_status,customer_status
                        FROM customer_stockpoint_cluster_assignment a
                        LEFT JOIN read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d 
                            ON d.customer_id = a.customer_id     
                        ''').df()
# stock_point_id	customer_id	h3_cell_id	customer_type	previous_cluster_id	assignment_type_id	
# assignment_type	contact_name	state_name	town_name	city_name	latitude	longitude	kyc_capture_status	customer_status
 

In [ ]:
customer_stockpoint_cluster_assignment_df.sample(2)

### 3. Cluster/Beat

In [ ]:
# stockpoint_h3_coverage_with_metadata.sample(2)
# stock_point_id	beat	beat_id	state_name	lga_name	ward_name	area_km2	confidence_level	latlng_coords	cluster_sp_dist_km	
# n_total_assigned_customers	n_assigned_active_customers	n_assigned_recent_activated_customers

with duckdb.connect(H3_DUCKDB_PATH) as conn: 
    stockpoint_h3_coverage_with_metadata = conn.execute("""
                       WITH CTE_Assignment_Summary AS(
                           SELECT 
                            stock_point_id, h3_cell_id as h3_cell, 
                            COUNT(DISTINCT customer_id) as n_total_assigned_customers,
                            CAST(SUM(CASE WHEN assignment_type_id = 1 THEN 1 ELSE 0 END) AS INT) AS n_assigned_active_customers,
                            CAST(SUM(CASE WHEN assignment_type_id = 3 THEN 1 ELSE 0 END) AS INT)  AS n_assigned_recent_activated_customers
                        FROM customer_stockpoint_cluster_assignment_df  
                        WHERE h3_cell_id NOT NULL
                        GROUP BY stock_point_id, h3_cell_id 
                       )    
                       SELECT 
                            c.stock_point_id, c.h3_cell as beat, primary_address_id as beat_id,
                            h.state_name, h.lga_name, h.ward_name, h.area_km2, h.confidence_level, h.latlng_json as latlng_coords,  
                            --c.cluster_centroid_lat, c.cluster_centroid_lng, 
                            c.cluster_sp_dist_km,
                            COALESCE(s.n_total_assigned_customers, 0) AS n_total_assigned_customers, 
                            COALESCE(s.n_assigned_active_customers, 0) AS n_assigned_active_customers, 
                            COALESCE(s.n_assigned_recent_activated_customers, 0) AS n_assigned_recent_activated_customers
                       FROM stockpoint_h3_coverage c
                       LEFT JOIN CTE_Assignment_Summary s ON c.stock_point_id = s.stock_point_id AND c.h3_cell  = s.h3_cell         
                       LEFT JOIN h3_cells h ON  c.h3_cell  = h.h3_index              
                        
                       """).df()
    
    
stockpoint_h3_coverage_with_metadata = gpd.GeoDataFrame(stockpoint_h3_coverage_with_metadata)
stockpoint_h3_coverage_with_metadata['latlng_coords'] = stockpoint_h3_coverage_with_metadata['latlng_coords'].apply(lambda x: ast.literal_eval(x))
stockpoint_h3_coverage_with_metadata = stockpoint_h3_coverage_with_metadata.merge(all_cluster_clipped_grid_list_df.rename(columns={'h3_cell': 'beat'}), 
                                                                                   on=['beat', 'stock_point_id'], how='left')
    

In [ ]:
stockpoint_h3_coverage_with_metadata.sample(2)

In [ ]:
test_sp_h3_coverage_with_metadata.info()

In [ ]:
test_sp_h3_coverage_with_metadata.iloc[0]['geometry'].__geo_interface__

In [ ]:
features = []
for _, row in test_sp_h3_coverage_with_metadata.iterrows():
    # Convert shapely geometry to GeoJSON
    geom = row['geometry'].__geo_interface__
    
# TypeError: the JSON object must be str, bytes or bytearray, not dict

-----------------------

# Errata

In [ ]:
#    # Load base data (processed)   
# # DB 
# db.truncate_insert_stockpoint_h3_coverage(sp_coverage_df_enhanced)
# db.upsert_customer_cluster_assignment(sp_assignment_df)
 
# # Data Required



In [ ]:
stockpoint_h3_coverage_with_metadata.sample(2)
# stock_point_id	beat	beat_id	state_name	lga_name	ward_name	area_km2	confidence_level	latlng_coords	cluster_sp_dist_km	
# n_total_assigned_customers	n_assigned_active_customers	n_assigned_recent_activated_customers

# Util Functions and Plotting

In [ ]:
test_spid = test_spids_contiguous[1]
geojson_boundary = sp_territories_dict[test_spid]['polygon']
test_sp_h3_coverage_with_metadata = stockpoint_h3_coverage_with_metadata.query(f'stock_point_id == {test_spid}')
test_sp_customer_assignment = customer_stockpoint_cluster_assignment_df.query(f'stock_point_id == {test_spid}').reset_index()

### 1. Plotting Bounday

In [ ]:
import folium 
geojson_boundary = sp_territories_dict[test_spid]['polygon']
m = folium.Map()
# Add to map with popups
m = folium.GeoJson(geojson_boundary).add_to(m)
m.save(test_plot_dir_boundary / f'test_territory_plot_{test_spid}.html')

### 2. Ploting grid

In [ ]:
import folium
import json
import folium
import json

def convert_sp_assigment_df_to_geojson(assignment_gdf, use_geometry = True):
    """
     
    """ 
    
    # Build GeoJSON FeatureCollection
    features = []
    for _, row in assignment_gdf.iterrows():
        # Convert latlng_coords to GeoJSON format (lng, lat order)
        coordinates = [[[coord[1], coord[0]] for coord in row['latlng_coords']]]
        
        # Create popup content
        popup_html = f"""
        <b>Beat ID:</b> {row['beat_id']}<br>
        <b>State:</b> {row['state_name']}<br>
        <b>LGA:</b> {row['lga_name']}<br>
        <b>Ward:</b> {row['ward_name']}<br>
        <b>Area:</b> {row['area_km2']:.2f} km²<br>
        <b>Distance to SP:</b> {row['cluster_sp_dist_km']:.2f} km<br>
        <b>Total Customers:</b> {row['n_total_assigned_customers']}<br>
        <b>Active Customers:</b> {row['n_assigned_active_customers']}<br>
        <b>Recently Activated Customers:</b> {row['n_assigned_recent_activated_customers']}
        """
        if use_geometry:
            # Convert shapely geometry to GeoJSON
            geom = row['geometry'].__geo_interface__
            
            feature = {
                "type": "Feature",
                "geometry": geom,
            } 
        else:
            feature = {
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": coordinates
                },
                "properties": {
                    "popup": popup_html,
                    "all_customers": row['n_total_assigned_customers'],
                    "active_customers": row['n_assigned_active_customers'],
                    "recently_activated_customers": row['n_assigned_recent_activated_customers']
                                    
                }
            }
        
        property ={"properties": {
                    "popup": popup_html,
                    "all_customers": row['n_total_assigned_customers'],
                    "active_customers": row['n_assigned_active_customers'],
                    "recently_activated": row['n_assigned_recent_activated_customers']
                }}
        feature.update(property) 
        # Append       
        features.append(feature)
        
    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }
    
    return geojson_data


def plot_h3_hexagons(sp_geojson, fill_col = 'all_customers', map_center=[6.5, 3.4]):
    """
    Efficiently plot H3 hexagons from GeoDataFrame with popups
    """
    # Create base map
    m = folium.Map(location=map_center, zoom_start=10)
    
    # Build GeoJSON FeatureCollection
    geojson_data = sp_geojson
    # Valid fill column name
    if fill_col in ["all_customers","active_customers","recently_activated_customers"]:
        fill_col = fill_col
    else:
        fill_col = 'all_customers'
    
    
    # Style function with color based on customer count
    def style_function_(feature):
        customers = feature['properties'][fill_col]

        if customers >= 200:
            color = '#800026'  # Dark red
        elif customers >= 150:
            color = '#BD0026'  # Red
        elif customers >= 100:
            color = '#E31A1C'  # Bright red
        elif customers >= 50:
            color = '#FC4E2A'  # Orange-red
        elif customers >= 20:
            color = '#FD8D3C'  # Orange
        elif customers >= 10:
            color = '#FEB24C'  # Light orange
        elif customers >= 5:
            color = '#FED976'  # Yellow
        elif customers >= 1:
            color = '#FFEDA0'  # Light yellow
        else:
            color = '#d9d9d9'  # Grey for 0

        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
    }
    
    def style_function_blue(feature):
        customers = feature['properties'][fill_col]
        if customers >= 200:
            color = '#08306b'
        elif customers >= 150:
            color = '#08519c'
        elif customers >= 100:
            color = '#2171b5'
        elif customers >= 50:
            color = '#4292c6'
        elif customers >= 20:
            color = '#6baed6'
        elif customers >= 10:
            color = '#9ecae1'
        elif customers >= 5:
            color = '#c6dbef'
        elif customers >= 1:
            color = '#deebf7'
        else:
            color = '#999999'
        
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    def style_function(feature):
        customers = feature['properties'][fill_col]
        if customers >= 200:
            color = '#67000d'
        elif customers >= 150:
            color = '#a50f15'
        elif customers >= 100:
            color = '#cb181d'
        elif customers >= 50:
            color = '#ef3b2c'
        elif customers >= 20:
            color = '#fb6a4a'
        elif customers >= 10:
            color = '#fc9272'
        elif customers >= 5:
            color = '#fcbba1'
        elif customers >= 1:
            color = '#fee0d2'
        else:
            color = '#999999'
        
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    # Add to map with popups
    folium.GeoJson(
        geojson_data,
        style_function=style_function,
        popup=folium.GeoJsonPopup(
            fields=['popup'],
            aliases=[''],
            labels=False
        )
    ).add_to(m)
    
    return m


def prepare_beat_folium_GeoJson(sp_geojson, fill_col = 'all_customers', use_blue=False):
    """
    Efficiently plot H3 hexagons from GeoDataFrame with popups
    """
    # Create base map 
    
    # Build GeoJSON FeatureCollection
    geojson_data = sp_geojson
    # Valid fill column name
    if fill_col in ["all_customers","active_customers","recently_activated_customers"]:
        fill_col = fill_col
    else:
        fill_col = 'all_customers'
    
    
    # Style function with color based on customer count    
    def style_function_blue(feature):
        customers = feature['properties'][fill_col]
        if customers >= 200:
            color = '#08306b'
        elif customers >= 150:
            color = '#08519c'
        elif customers >= 100:
            color = '#2171b5'
        elif customers >= 50:
            color = '#4292c6'
        elif customers >= 20:
            color = '#6baed6'
        elif customers >= 10:
            color = '#9ecae1'
        elif customers >= 5:
            color = '#c6dbef'
        elif customers >= 1:
            color = '#deebf7'
        else:
            color = '#999999'
        
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    def style_function_red(feature):
        customers = feature['properties'][fill_col]
        if customers >= 200:
            color = '#67000d'
        elif customers >= 150:
            color = '#a50f15'
        elif customers >= 100:
            color = '#cb181d'
        elif customers >= 50:
            color = '#ef3b2c'
        elif customers >= 20:
            color = '#fb6a4a'
        elif customers >= 10:
            color = '#fc9272'
        elif customers >= 5:
            color = '#fcbba1'
        elif customers >= 1:
            color = '#fee0d2'
        else:
            color = '#999999'
        
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    if use_blue:
        use_style_function = style_function_blue
    else:
        use_style_function = style_function_red
    # Add to map with popups
    beat_folium_GeoJson = folium.GeoJson(
        geojson_data,
        style_function=use_style_function,
        popup=folium.GeoJsonPopup(
            fields=['popup'],
            aliases=[''],
            labels=False
        )
    ) 
    
    return beat_folium_GeoJson



In [ ]:

# Usage
ftypes = ["all_customers","active_customers","recently_activated_customers"]
ftype = ftypes[0]
use_geometry = True

suff = 'clipped' if use_geometry else 'valina'
filename = test_plot_dir_grid_coverage / f'_sp-hex-coverage-{test_spid}_{suff}_{ftype}.html'
 
sp_beat_geojson = convert_sp_assigment_df_to_geojson(test_sp_h3_coverage_with_metadata, use_geometry= use_geometry)
m = plot_h3_hexagons(sp_beat_geojson, fill_col=ftype)
m.save(filename)

### 3. Plotting Customers

In [ ]:
# test_sp_customer_assignment.sample(1)
# test_sp_customer_assignment.customer_type.unique() # ['buying customers', 'recently activated']
# test_sp_customer_assignment.customer_status.unique()

In [ ]:
def convert_customers_to_geojson(customer_df):
    """
    Convert customer dataframe to GeoJSON points for scatter plotting
    """
    features = []
    for _, row in customer_df.iterrows():
        popup_html = f"""
        <b>Customer ID:</b> {row['customer_id']}<br>
        <b>Contact:</b> {row['contact_name']}<br>
        <b>Type:</b> {row['customer_type']}<br>
        <b>Assignment Type:</b> {row['assignment_type']}<br>
        <b>State:</b> {row['state_name']}<br>
        <b>City:</b> {row['city_name']}<br>
        <b>Status:</b> {row['customer_status']}<br>
        <b>KYC Status:</b> {row['kyc_capture_status']}
        """
        
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row['longitude'], row['latitude']]
            },
            "properties": {
                "popup": popup_html,
                "customer_type": row['customer_type'],
                "assignment_type_id": row['assignment_type_id'],
                "customer_status": row['customer_status'],
                "stock_point_id": row['stock_point_id']
            }
        }
        features.append(feature)
    
    return {
        "type": "FeatureCollection",
        "features": features
    }


def prepare_customer_assignment_GeoJson(geojson_data):
    """
    Plot customers as scatter points
    """ 
    
    # Style function based on customer type/status
    def style_function(feature):
        customer_type = feature['properties']['customer_type']
        status = feature['properties']['customer_status']
        
        # Color by customer type
        if customer_type == 'buying customers':
            color = "#850505a6"
        elif customer_type == 'recently activated':
            color = "#43016e"
        else:
            color = "#8c8f91"
            
        # Adjust opacity by status
        opacity = 0.8 if status == 'Active' else 0.4
        
        return {
            'fillColor': color,
            'color': color,
            'weight': 2,
            'fillOpacity': opacity,
            'radius': 2
        }
    
    # Use CircleMarker for better performance with many points
    return folium.GeoJson(
        geojson_data,
        marker=folium.CircleMarker(),
        style_function=style_function,
        popup=folium.GeoJsonPopup(
            fields=['popup'],
            aliases=[''],
            labels=False
        )
    )  

In [ ]:
# customer_stockpoint_cluster_assignment_df.customer_type.unique()
# ['stock_point_id', 'customer_id', 'h3_cell_id', 'customer_type',
# 'previous_cluster_id', 'assignment_type_id', 'assignment_type',
# 'contact_name', 'state_name', 'town_name', 'city_name', 'latitude',
# 'longitude', 'kyc_capture_status', 'customer_status']

customer_geojson = convert_customers_to_geojson(test_sp_customer_assignment)
customer_geojson_folium = prepare_customer_assignment_GeoJson(customer_geojson)

mm = folium.Map(location=[6.47, 3.35])

customer_geojson_folium.add_to(mm)

filename = test_plot_dir_customer_assignment / f'_sp-customer-assignment-{test_spid}.html'
mm.save(filename)

### PLOT MODULE

In [ ]:
processed_sp_dim_df.head(1)

In [ ]:
test_spid = test_spids_contiguous[1]
test_sp_dim = processed_sp_dim_df.query(f'stock_point_id == {test_spid}').reset_index()
test_sp_h3_coverage_with_metadata = stockpoint_h3_coverage_with_metadata.query(f'stock_point_id == {test_spid}').reset_index()
test_sp_customer_assignment = customer_stockpoint_cluster_assignment_df.query(f'stock_point_id == {test_spid}').reset_index()

fill_cols = ["all_customers","active_customers","recently_activated_customers"]
fill_col = fill_cols[0]
use_geometry = True
suff = 'clipped' if use_geometry else 'valina'
plot_filename = test_plot_dir / f'viz_{test_spid}_{suff}_{fill_col}.html'

# Plot Elements 
coord_lat, coord_lng = test_sp_dim['latitude'].iloc[0], test_sp_dim['longitude'].iloc[0]
spname = test_sp_dim['stock_point_name'].iloc[0]
sp_boundary_geojson = sp_territories_dict[test_spid]['polygon']
sp_beat_geojson = convert_sp_assigment_df_to_geojson(test_sp_h3_coverage_with_metadata, use_geometry= use_geometry)
customer_geojson = convert_customers_to_geojson(test_sp_customer_assignment)


In [ ]:
from folium import Map, FeatureGroup, LayerControl, Marker
from folium.plugins import FeatureGroupSubGroup

# 1. Base Map
base_map = folium.Map(location = [coord_lat, coord_lng],
                      tiles="CartoDB Positron",
                      # tiles="OpenStreetMap",
                      prefer_canvas = True
                      )


fg_parent = FeatureGroup(name=spname) 

## 2. Add Boundary File
fg_boundary = FeatureGroupSubGroup(fg_parent, name="Boundary")
# fg_boundary = folium.FeatureGroup(name="Boundary")
folium.GeoJson(sp_boundary_geojson).add_to(fg_boundary)


## 3. Beats and Assignments
fg_beats = FeatureGroupSubGroup(fg_parent, name="Beats and Assignments")
# fg_beats = folium.FeatureGroup(name="Beats and Assignments")
beat_folium_GeoJson = prepare_beat_folium_GeoJson(sp_beat_geojson, fill_col = fill_col)
beat_folium_GeoJson.add_to(fg_beats)


## 3b. Buying Customers
fg_customers = FeatureGroupSubGroup(fg_parent,name="Customers")
# fg_customers = folium.FeatureGroup(name="Customers")
customer_geojson_folium = prepare_customer_assignment_GeoJson(customer_geojson)

customer_geojson_folium.add_to(fg_customers) 

# 4,. Marker
fg_spmarker = FeatureGroupSubGroup(fg_parent,name="Stock Point")
# fg_spmarker = folium.FeatureGroup(name="Stock Point")
folium.Marker(
    location=[coord_lat, coord_lng],
    popup=spname,
    icon=folium.Icon(color="green")
).add_to(fg_spmarker)

# # ------------------------
# # Add FeatureGroups to map
# # ------------------------

# fg_boundary.add_to(base_map)
# fg_beats.add_to(base_map)
# fg_customers.add_to(base_map)
# fg_spmarker.add_to(base_map)
fg_parent.add_to(base_map)

# Add LayerControl
folium.LayerControl(collapsed=True).add_to(base_map)

base_map.save(plot_filename)

In [ ]:
import folium
from folium import Map, FeatureGroup, LayerControl, Marker

def create_stockpoint_map(spid, processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, 
                         customer_stockpoint_cluster_assignment_df, sp_territories_dict,
                         fill_col="all_customers", use_geometry=True, output_dir=None):
    """
    Create an interactive folium map for a stock point with territories, beats, and customers.
    
    Parameters:
    -----------
    spid : int
        Stock point ID to visualize
    processed_sp_dim_df : DataFrame
        Stock point dimension data with coordinates and names
    stockpoint_h3_coverage_with_metadata : DataFrame
        H3 coverage data with metadata for beats
    customer_stockpoint_cluster_assignment_df : DataFrame
        Customer assignment data
    sp_territories_dict : dict
        Dictionary containing territory polygons by stock point ID
    fill_col : str, default "all_customers"
        Column to use for fill color in beats visualization
    use_geometry : bool, default True
        Whether to use clipped geometry
    output_dir : Path, optional
        Directory to save the HTML file. If None, returns map object only
    
    Returns:
    --------
    folium.Map or str
        Returns folium map object if output_dir is None, otherwise saves file and returns filename
    """
    
    # Prepare data
    sp_dim = processed_sp_dim_df.query(f'stock_point_id == {spid}').reset_index()
    sp_h3_coverage = stockpoint_h3_coverage_with_metadata.query(f'stock_point_id == {spid}').reset_index()
    sp_customer_assignment = customer_stockpoint_cluster_assignment_df.query(f'stock_point_id == {spid}').reset_index()
    
    # Extract plot elements
    coord_lat, coord_lng = sp_dim['latitude'].iloc[0], sp_dim['longitude'].iloc[0]
    spname = sp_dim['stock_point_name'].iloc[0]
    sp_boundary_geojson = sp_territories_dict[spid]['polygon']
    sp_beat_geojson = convert_sp_assigment_df_to_geojson(sp_h3_coverage, use_geometry=use_geometry)
    customer_geojson = convert_customers_to_geojson(sp_customer_assignment)
    
    # Create base map
    base_map = folium.Map(location=[coord_lat, coord_lng],
                          tiles="CartoDB Positron",
                          prefer_canvas=True,
                          zoom_start=10)
    
    # Add boundary layer
    fg_boundary = FeatureGroup(name="Boundary")
    folium.GeoJson(sp_boundary_geojson).add_to(fg_boundary)
    
    # Add beats layer
    fg_beats = FeatureGroup(name="Beats and Assignments") 
    beat_folium_GeoJson = prepare_beat_folium_GeoJson(sp_beat_geojson, fill_col=fill_col)
    beat_folium_GeoJson.add_to(fg_beats)
    
    # Add customers layer
    fg_customers = FeatureGroup(name="Customers")
    customer_geojson_folium = prepare_customer_assignment_GeoJson(customer_geojson)
    customer_geojson_folium.add_to(fg_customers)
    
    # Add stock point marker
    fg_spmarker = FeatureGroup(name="Stock Point")
    folium.Marker(
        location=[coord_lat, coord_lng],
        popup=spname,
        icon=folium.Icon(color="green")
    ).add_to(fg_spmarker)
    
    # Add all layers to map
    fg_boundary.add_to(base_map)
    fg_beats.add_to(base_map)
    fg_customers.add_to(base_map)
    fg_spmarker.add_to(base_map)
    
    # Add layer control
    folium.LayerControl().add_to(base_map)
    
    # Save or return map
    if output_dir:
        suff = 'clipped' if use_geometry else 'vanilla'
        plot_filename = output_dir / f'viz_{spid}_{suff}_{fill_col}.html'
        base_map.save(plot_filename)
        return str(plot_filename)
    else:
        return base_map

# Usage example:
map_obj = create_stockpoint_map(test_spids_noncontiguous[0], processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, 
                               customer_stockpoint_cluster_assignment_df, sp_territories_dict,
                               fill_col="all_customers", output_dir=test_plot_dir)

In [ ]:
# help(folium.Map)
# help(folium.Marker)

### EDA

In [ ]:
### Customer Details


with duckdb.connect(H3_DUCKDB_PATH) as conn: 
    df__ = conn.execute('''
    SELECT 
        * 
        ----customer_id,  contact_name,state_name, town_name, city_name, latitude, longitude, kyc_capture_status,customer_status
    --- FROM read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d ---ON d.customer_id = a.customer_id             
    FROM customer_stockpoint_cluster_assignment 
    LIMIT 10 
    '''
    ).df()
    
df__.columns 
df__.sample(3)    

# ['customer_id', 'business_id', 'created_date', 'contact_name',
# 'contact_phone', 'state_name', 'town_name', 'city_name', 'latitude',
# 'longitude', 'customer_status', 'status', 'first_name',
# 'is_location_captured', 'is_location_verified', 'kyc_capture_status',
# 'agent_id', 'agent_name', 'address']

In [ ]:
## Prepare a cluster summary for each stockpoint
"""
1. Using stockpoint h3 coverage 
2. For each h3_cell/beats get the grid address  
3. Count of customers [Total Assigned, Active and recently Activated]
4. Cluster distance to SP
5. Add Polygon for easy ploting
"""



with duckdb.connect(H3_DUCKDB_PATH) as conn: 
    stockpoint_h3_coverage_with_metadata = conn.execute("""
                       WITH CTE_Assignment_Summary AS(
                           SELECT 
                            stock_point_id, h3_cell_id as h3_cell, 
                            COUNT(DISTINCT customer_id) as n_total_assigned_customers,
                            CAST(SUM(CASE WHEN assignment_type_id = 1 THEN 1 ELSE 0 END) AS INT) AS n_assigned_active_customers,
                            CAST(SUM(CASE WHEN assignment_type_id = 3 THEN 1 ELSE 0 END) AS INT)  AS n_assigned_recent_activated_customers
                        FROM customer_stockpoint_cluster_assignment_df  
                        WHERE h3_cell_id NOT NULL
                        GROUP BY stock_point_id, h3_cell_id 
                       )    
                       SELECT 
                            c.stock_point_id, c.h3_cell as beat, primary_address_id as beat_id,
                            h.state_name, h.lga_name, h.ward_name, h.area_km2, h.confidence_level, h.latlng_json as latlng_coords,
                            --c.cluster_centroid_lat, c.cluster_centroid_lng, 
                            c.cluster_sp_dist_km,
                            COALESCE(s.n_total_assigned_customers, 0) AS n_total_assigned_customers, 
                            COALESCE(s.n_assigned_active_customers, 0) AS n_assigned_active_customers, 
                            COALESCE(s.n_assigned_recent_activated_customers, 0) AS n_assigned_recent_activated_customers
                       FROM stockpoint_h3_coverage c
                       LEFT JOIN CTE_Assignment_Summary s ON c.stock_point_id = s.stock_point_id AND c.h3_cell  = s.h3_cell         
                       LEFT JOIN h3_cells h ON  c.h3_cell  = h.h3_index         
                       """).df()
    
# stockpoint_h3_coverage_with_metadata

In [ ]:
df_.columns

In [ ]:
stockpoint_h3_coverage_df.sample(4) 

In [ ]:

'''
Customers
1. Assigned Active/Buying --- 1
2. Unassigned Active/Buying --- 2
3. Assigned Recently Activated --- 3
'''


print(customer_stockpoint_cluster_assignment_df.value_counts(subset=['assignment_tier','customer_type']))
print(customer_stockpoint_cluster_assignment_df.value_counts(subset=['assignment_type']))
customer_stockpoint_cluster_assignment_df.sample(4) #53015
# customer_stockpoint_cluster_assignment_df.value_counts(subset=['status'])



In [ ]:
36759 + 16256

In [ ]:
customer_stockpoint_cluster_assignment_df.columns

In [ ]:
sample_grid_result1 = cluster_results.get('grid_results').get(test_spids_contiguous[0])
sample_grid_result2 = cluster_results.get('grid_results').get(test_spids_noncontiguous[0])

# RUN

In [3]:
%load_ext autoreload
%autoreload 2 

from src.data.load_postprocessed_data import load_data, postprocess_map_data
from src.viz.viz import create_stockpoint_map
from config.settings import OUTPUT_DIR

plot_dir = OUTPUT_DIR / 'map/sp-clustering-and-assignment'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
postprocess_map_data(return_data = False, from_local=False)

Preparing data from scratch...
Using clustering results file: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports/clustering/ALL_SPS_CLUSTER_R8_2025-08-22.pickle
Saving processed data to local directory...
Saving processed data to cicd directory...


In [2]:
test_spids_contiguous = ['1647131', '1647113','1647127']
test_spids_noncontiguous = ['1646971','1647136','1647376','1647380']

In [ ]:
processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, customer_stockpoint_cluster_assignment_df, sp_territories_dict = load_data(from_local=True)

In [ ]:
 

# sp_boundary_geojson 

<class 'shapely.geometry.polygon.Polygon'>


In [4]:
# test_spids_noncontiguous+test_spids_noncontiguous

# processed_sp_dim_df

In [22]:
# for spid in test_spids_contiguous+test_spids_noncontiguous: 
for spid in processed_sp_dim_df.stock_point_id.values: 
    spid = str(spid)
    print(spid)
    try:
        map_obj = create_stockpoint_map(spid, processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, 
                                customer_stockpoint_cluster_assignment_df, sp_territories_dict,
                                fill_col="all_customers",use_geometry=True, output_dir=plot_dir)
    except Exception as e:
        print(f"""Can't Map: {e}""")
    
    

1647128
Can't Map: '1647128'
1647136
1648082
1648081
1647076
Can't Map: '1647076'
1647394
1647396
1647442
1647075
Can't Map: '1647075'
1647437
1647077
Can't Map: '1647077'
1647400
1648086
Can't Map: '1648086'
1647081
1647438
1647419
1647403
1647424
1647398
1647401
1647402
1647421
1647422
1646941
1647425
1647436
1647434
1647420
1646945
Can't Map: '1646945'
1646999
1647010
1646991
1647024
1646989
Can't Map: '1646989'
1647033
1646976
Can't Map: '1646976'
1646971
1647011
Can't Map: '1647011'
1647050
1647006
Can't Map: '1647006'
1646995
1647391
1647372
1647377
1647387
1647062
1647381
1647345
1647376
1647353
1648083
Can't Map: '1648083'
1647347
1647341
1647350
1647125
1647107
Can't Map: '1647107'
1647122
1647113
1647132
1647124
1647380
1647126
1647109
1647371
1647131
1647110
1647137
1647106
1647115
Can't Map: '1647115'
1647127
1647130
1647382
1647108
1647443
Can't Map: '1647443'
1647141
1647187


In [21]:
processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, customer_stockpoint_cluster_assignment_df, sp_territories_dict = load_data(from_local=True)

Loading data from local pickle files...


In [14]:
import os
import glob
from config.settings import EXPORTS_DIR 

def get_latest_clusterfile():
    # Define the directory path
    cluster_results_dir = EXPORTS_DIR / 'clustering' 

    # Expand the user directory shortcut
    expanded_path = os.path.expanduser(cluster_results_dir)

    # Use glob to find all .pickle files
    list_of_files = glob.glob(f'{expanded_path}/*.pickle')

    # Check if any .pickle files were found
    if not list_of_files:
        print("No .pickle files found in the specified directory.")
        return None
    else:
        # Find the latest file by modification time
        latest_file = max(list_of_files, key=os.path.getmtime)
        # print(f"The latest .pickle file is: {latest_file}")
        return latest_file
    
get_latest_clusterfile()    

'/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports/clustering/ALL_SPS_CLUSTER_R8_2025-08-22.pickle'

In [18]:
from pathlib import Path
Path('/home/bt/project/Insight_and_Discovery/Clustering_And_Routes/data')

PosixPath('/home/bt/project/Insight_and_Discovery/Clustering_And_Routes/data')

In [ ]:
# processed_sp_dim_df.info()

#  #   Column            Non-Null Count  Dtype  
# ---  ------            --------------  -----  
#  0   stock_point_id    76 non-null     int64  
#  1   stock_point_name  76 non-null     object 
#  2   latitude          76 non-null     float64
#  3   longitude         76 non-null     float64
 
 
#  stockpoint_h3_coverage_with_metadata.info()

#  #   Column                                 Non-Null Count   Dtype   
# ---  ------                                 --------------   -----   
#  0   stock_point_id                         136710 non-null  int64   
#  1   beat                                   136710 non-null  object  
#  2   beat_id                                136710 non-null  object  
#  3   state_name                             136710 non-null  object  
#  4   lga_name                               136710 non-null  object  
#  5   ward_name                              136710 non-null  object  
#  6   area_km2                               136710 non-null  float64 
#  7   confidence_level                       136710 non-null  object  
#  8   latlng_coords                          136710 non-null  object  
#  9   cluster_sp_dist_km                     136710 non-null  float64 
#  10  n_total_assigned_customers             136710 non-null  int64   
#  11  n_assigned_active_customers            136710 non-null  int32   
#  12  n_assigned_recent_activated_customers  136710 non-null  int32   
#  13  geometry                               136354 non-null  geometry
 
 
#  customer_stockpoint_cluster_assignment_df.info()
 
#  #   Column               Non-Null Count  Dtype  
# ---  ------               --------------  -----  
#  0   stock_point_id       60557 non-null  int64  
#  1   customer_id          60557 non-null  int64  
#  2   h3_cell_id           53845 non-null  object 
#  3   customer_type        60557 non-null  object 
#  4   previous_cluster_id  0 non-null      object 
#  5   assignment_type_id   60557 non-null  int32  
#  6   assignment_type      60557 non-null  object 
#  7   contact_name         60557 non-null  object 
#  8   state_name           60557 non-null  object 
#  9   town_name            51082 non-null  object 
#  10  city_name            60430 non-null  object 
#  11  latitude             60557 non-null  float64
#  12  longitude            60557 non-null  float64
#  13  kyc_capture_status   60557 non-null  object 
#  14  customer_status      60557 non-null  object  
 
#  sp_territories_dict['1646941']#.info()

# {'polygon': <POLYGON ((3.989 7.214, 3.991 7.185, 3.995 7.181, 3.997 7.176, 3.989 7.145, ...>,
#  'lga_ids': [1644, 1648, 1659],
#  'is_contiguous': True,
#  'sub_territories': [<POLYGON ((3.989 7.214, 3.991 7.185, 3.995 7.181, 3.997 7.176, 3.989 7.145, ...>],
#  'total_area_km2': 1423.1352,
#  'territory_version': 'v1.2',
#  'lga_count': 3,
#  'validation_status': 'valid'} 
 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   stock_point_id    76 non-null     int64  
 1   stock_point_name  76 non-null     object 
 2   latitude          76 non-null     float64
 3   longitude         76 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 2.5+ KB
